In [1]:
import os
import torch
import librosa

# 🛡️ Tự động kiểm tra & cài đặt pyctcdecode nếu phiên làm việc chưa có (nhớ bật Internet: ON bên cột Settings Kaggle)
try:
    from pyctcdecode import build_ctcdecoder
except ModuleNotFoundError:
    print("⏳ Đang tự động cài đặt pyctcdecode và kenlm (dùng --no-deps)...")
    !pip install -q --no-deps pyctcdecode pygtrie hypothesis rapidfuzz https://github.com/kpu/kenlm/archive/master.zip
    from pyctcdecode import build_ctcdecoder

# 🔑 Tự động nạp HuggingFace Token từ Kaggle Secrets (HF_TOKEN_READ)
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    hf_token = UserSecretsClient().get_secret("HF_TOKEN_READ")
    login(token=hf_token)
    print("🔑 Đã xác thực HuggingFace Token (HF_TOKEN_READ) thành công!")
except Exception as e:
    print("ℹ️ Chưa nạp secret HF_TOKEN_READ hoặc đang chạy ngoài Kaggle, sẽ tải ở chế độ ẩn danh.")

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

# 1. Khởi tạo mô hình Acoustic chính xác (Fine-tuned 250h cho ASR)
MODEL_ID = "nguyenvulebinh/wav2vec2-base-vietnamese-250h"
print(f"⏳ Đang tải mô hình ASR: {MODEL_ID}...")
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).to("cuda")

# 2. Đọc danh mục 6,069 từ khóa thuốc từ file drugs.txt
drugs_path = "/kaggle/input/datasets/hdtuznn/sppr-text/text/drugs.txt"
if os.path.exists(drugs_path):
    hotwords_list = [w.strip() for w in open(drugs_path, encoding="utf-8") if w.strip()]
    print(f"✅ Đã load thành công {len(hotwords_list):,} từ khóa thuốc vào danh sách Hotwords!")
else:
    print("⚠️ Chưa tìm thấy file drugs.txt, kiểm tra lại đường dẫn!")
    hotwords_list = []

# 3. Nạp KenLM binary + Hotwords vào pyctcdecode
kenlm_bin_path = "/kaggle/input/models/hdtuznn/kenlm/pytorch/default/1/kenlm_vi_medical_4gram.bin"
print(f"⏳ Đang nạp bộ giải mã Beam Search với KenLM: {kenlm_bin_path}...")

decoder_stage1 = build_ctcdecoder(
    labels=list(processor.tokenizer.get_vocab().keys()),  # 96 ký tự khớp lm_head
    kenlm_model_path=kenlm_bin_path
)
print("✅ Nạp thành công trọn bộ giải mã Tầng 1!")

# 4. Hàm giải mã chuẩn Tầng 1 (Dùng để lấy ASR Predicted Question đưa sang ViT5)
def decode_stage1_asr(audio_path):
    speech, sr = librosa.load(audio_path, sr=16000)
    inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True).input_values.to("cuda")
    with torch.no_grad():
        logits = model(inputs).logits[0].cpu().numpy()
    
    trans = decoder_stage1.decode(
        logits,
        beam_width=16,
        beam_prune_logp=-5.0,
        token_min_logp=-3.0,
        hotwords=hotwords_list,
        hotword_weight=15.0
    )
    return trans.strip()

print("\n🎉 SẴN SÀNG! Bạn có thể gọi decode_stage1_asr(audio_path) trong các vòng lặp tiếp theo!")

⏳ Đang tự động cài đặt pyctcdecode và kenlm (dùng --no-deps)...
     \ 553.6 kB 4.4 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.4 MB/s eta 0:00:00:00:01
🔑 Đã xác thực HuggingFace Token (HF_TOKEN_READ) thành công!
⏳ Đang tải mô hình ASR: nguyenvulebinh/wav2vec2-base-vietnamese-250h...


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/213 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


⚠️ Chưa tìm thấy file drugs.txt, kiểm tra lại đường dẫn!
⏳ Đang nạp bộ giải mã Beam Search với KenLM: /kaggle/input/models/hdtuznn/kenlm/pytorch/default/1/kenlm_vi_medical_4gram.bin...
✅ Nạp thành công trọn bộ giải mã Tầng 1!

🎉 SẴN SÀNG! Bạn có thể gọi decode_stage1_asr(audio_path) trong các vòng lặp tiếp theo!


In [2]:
import os
import re
import json
import random
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

# Set random seed for reproducibility
random.seed(42)

def simulate_vietnamese_asr_errors(text, drug_set):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove punctuation
    text = re.sub(r'[\,\.\?\!\;\:\-\"\'\(\)]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    words = text.split()
    noisy_words = []
    
    phonetic_swaps = {'tr': 'ch', 'ch': 'tr', 's': 'x', 'x': 's', 'l': 'n', 'n': 'l', 'r': 'd', 'd': 'r', 'gi': 'd'}
    phonetic_endings = {'nh': 'n', 'n': 'nh', 'ng': 'n', 'c': 't', 't': 'c'}
    
    for w in words:
        if w in drug_set and random.random() < 0.3:
            if len(w) > 3:
                if w.endswith('l'): w = w[:-1] + 'n'
                elif w.endswith('m'): w = w[:-1] + 'n'
                elif 'c' in w: w = w.replace('c', 'k', 1)
        elif random.random() < 0.15 and len(w) > 2:
            for prefix, replacement in phonetic_swaps.items():
                if w.startswith(prefix):
                    w = replacement + w[len(prefix):]
                    break
            else:
                for suffix, replacement in phonetic_endings.items():
                    if w.endswith(suffix):
                        w = w[:-len(suffix)] + replacement
                        break
        noisy_words.append(w)
        
    noisy_text = " ".join(noisy_words)
    return f"fix_asr: {noisy_text}"

randomqa_path = "/kaggle/input/datasets/hdtuznn/clean-questioned/cleaned_questions.txt"
drugs_path = "/kaggle/input/datasets/hdtuznn/sppr-text/data/text/drugs.txt"

if os.path.exists(randomqa_path):
    print(f"[INFO] Reading raw QA file from: {randomqa_path}")
    with open(randomqa_path, "r", encoding="utf-8") as f:
        raw_lines = [line.strip() for line in f if line.strip()]
    
    drug_set = set()
    if os.path.exists(drugs_path):
        with open(drugs_path, "r", encoding="utf-8") as f:
            drug_set = set(line.strip().lower() for line in f if line.strip())
        print(f"[INFO] Loaded {len(drug_set):,} drug keywords from {drugs_path}")
        
    print(f"[INFO] Simulating ASR errors for {len(raw_lines):,} medical sentences...")
    dataset_records = []
    
    for line in tqdm(raw_lines, desc="Processing samples"):
        target_text = line
        input_text = simulate_vietnamese_asr_errors(target_text, drug_set)
        dataset_records.append({
            "input_text": input_text,
            "target_text": target_text
        })
        
    df_dataset = pd.DataFrame(dataset_records).sample(frac=1.0, random_state=42).reset_index(drop=True)
    split_idx = int(len(df_dataset) * 0.95)
    
    df_train = df_dataset.iloc[:split_idx]
    df_val = df_dataset.iloc[split_idx:]
    
    train_json_path = "/kaggle/working/train_vit5.json"
    val_json_path = "/kaggle/working/val_vit5.json"
    
    df_train.to_json(train_json_path, orient="records", force_ascii=False, indent=2)
    df_val.to_json(val_json_path, orient="records", force_ascii=False, indent=2)
    
    print("[SUCCESS] Completed building ViT5 Rewrite dataset.")
    print(f"[INFO] Train set: {len(df_train):,} samples -> {train_json_path}")
    print(f"[INFO] Val set:   {len(df_val):,} samples -> {val_json_path}")
    
    print("\n[INFO] Sample records from train set:")
    for _, row in df_train.head(3).iterrows():
        print("-" * 80)
        print(f"Input  (Simulated ASR) : {row['input_text']}")
        print(f"Target (Clean Medical) : {row['target_text']}")
else:
    print(f"[ERROR] Cannot find file: {randomqa_path}")

[INFO] Reading raw QA file from: /kaggle/input/datasets/hdtuznn/clean-questioned/cleaned_questions.txt
[INFO] Loaded 6,069 drug keywords from /kaggle/input/datasets/hdtuznn/sppr-text/data/text/drugs.txt
[INFO] Simulating ASR errors for 67,365 medical sentences...


Processing samples:   0%|          | 0/67365 [00:00<?, ?it/s]

[SUCCESS] Completed building ViT5 Rewrite dataset.
[INFO] Train set: 63,996 samples -> /kaggle/working/train_vit5.json
[INFO] Val set:   3,369 samples -> /kaggle/working/val_vit5.json

[INFO] Sample records from train set:
--------------------------------------------------------------------------------
Input  (Simulated ASR) : fix_asr: người bệnh đang trong đợt cấp của trào lgược dạ dày thực quảnh erosive đã được chỉ định sanaperon 20mg nếu sau 4 tuần triệu chứng ợ nóng và nuốt khó đã hết hoàn toàn chiến lược điều trị tiếp theo có thay đổi gì khác biệt so với phát đồ dùng sanaperol liên tục 8 tuần cho bệnh lhân còn lại
Target (Clean Medical) : Người bệnh đang trong đợt cấp của trào ngược dạ dày thực quản erosive, đã được chỉ định Sanaperol 20mg. Nếu sau 4 tuần, triệu chứng ợ nóng và nuốt khó đã hết hoàn toàn, chiến lược điều trị tiếp theo có thay đổi gì khác biệt so với phác đồ dùng Sanaperol liên tục 8 tuần cho bệnh nhân còn lại?
-------------------------------------------------------